In [11]:
%pip install boto3

python(81732) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


You should consider upgrading via the '/Users/akshaypatade/Desktop/Projects/purchase-orders/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [23]:
import boto3
import json
import os

from dotenv import load_dotenv

load_dotenv()

textract = boto3.client('textract', aws_access_key_id = os.environ.get("AWS_ACCESS_KEY_ID"), aws_secret_access_key = os.environ.get("AWS_SECRET_ACCESS_KEY"), region_name = os.environ.get("AWS_DEFAULT_REGION"))

In [40]:
def extract_product_table(file_path):
    with open(file_path, 'rb') as file:
        # Call Textract
        response = textract.analyze_document(
            Document={'Bytes': file.read()},
            FeatureTypes=['TABLES']
        )
    
    # Map Block Ids
    blocks = {block['Id']: block for block in response['Blocks']}

    # Helper function to extract text from a block's relationships
    def get_text_from_relationships(relationships):
        text = []
        for rel in relationships:
            for word_id in rel['Ids']:
                if blocks[word_id]['BlockType'] == 'WORD':
                    text.append(blocks[word_id]['Text'])
        return ' '.join(text)

    # Extract all tables
    product_table = []
    for block in response['Blocks']:
        if block['BlockType'] == 'TABLE':
            rows = []
            for relationship in block.get('Relationships', []):
                if relationship['Type'] == 'CHILD':
                    for cell_id in relationship['Ids']:
                        cell = blocks[cell_id]
                        if cell['BlockType'] == 'CELL':
                            row_index = cell['RowIndex']
                            col_index = cell['ColumnIndex']
                            text = get_text_from_relationships(cell.get('Relationships', []))
                            while len(rows) < row_index:
                                rows.append([])
                            while len(rows[row_index - 1]) < col_index:
                                rows[row_index - 1].append("")
                            rows[row_index - 1][col_index - 1] = text

            # Check for product-specific headers
            if rows and any(header in rows[0] for header in ["Product Code", "Description", "Quantity", "Amount", "Qty", "Price",  "Unit Price", "Request Item", "Manufacturer Code", "Unit Cost"]):
                product_table = rows
                break  # Stop after finding the first matching table

    return product_table


In [36]:
def clean_table(table):
    # Remove rows where all elements are empty
    cleaned_table = [row for row in table if any(cell.strip() for cell in row)]
    return cleaned_table

In [46]:
# Example Usage
pdf_file = 'Hard -3.pdf'
product_table = extract_product_table(pdf_file)

# Display the product table
if product_table:
    print("Product Table:")
    cleaned_table = clean_table(product_table)
    for row in cleaned_table:
        print(row)
else:
    print("No product table found.")


Product Table:
['Qty', 'Item #', 'Item Description', 'Vendor Part #', 'Cost', 'Ext Cost']
['70', '6268188', 'Steel Bolt M6 50mm Galvanized Fine', '-', '$41.71', '$2919.70']
['1', '6268188', 'Titanium Nut 1/4" 50mm Uncoated Fine', '-', '$42.13', '$42.13']
['14', '6268188', 'Brass Stud 1/4" 40mm Uncoated Coarse', '-', '$41.71', '$583.94']
['', '', '', '', '', 'Freight: $0.00']
['', '', '', '', '', 'Total: $3545.77']
